# 05. Отраслевой анализ (ОКВЭД)

> **Краткое резюме для руководителя**: Этот ноутбук анализирует межотраслевые денежные потоки. Матрица ОКВЭД×ОКВЭД показывает, какие отрасли торгуют между собой и на какие суммы. Тепловая карта (heatmap) визуализирует эти потоки. Дополнительно выявляются «кросс-отраслевые хабы» — компании, работающие с контрагентами из множества разных отраслей (что может указывать на торговые дома, посредников или фиктивные компании).

**Процесс**:
1. Загружаем анализированный граф из pickle
2. Строим матрицу ОКВЭД×ОКВЭД (оборот между отраслями)
3. Визуализируем как тепловую карту
4. Вычисляем ОКВЭД-разнообразие для каждого узла
5. Выявляем кросс-отраслевые хабы

**Предпосылки**: Запустите `03_graph_analysis.ipynb`.

---

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.analysis import build_okved_matrix, compute_okved_diversity

## Загрузка графа

In [ ]:
graph_path = config.OUTPUT_FILTERED_GRAPH_PICKLE
print(f'Loading graph from: {graph_path}')

with open(graph_path, 'rb') as f:
    G = pickle.load(f)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Распределение ОКВЭД-кодов по узлам
okved_counts = pd.Series([d.get('okved_code', '00') for _, d in G.nodes(data=True)])
okved_counts = okved_counts[okved_counts != '00'].value_counts()
print(f'\nNodes with OKVED codes (excl. "00"): {okved_counts.sum()}')
print(f'Unique OKVED codes: {len(okved_counts)}')
print(f'\nTop-10 OKVED codes by node count:')
print(okved_counts.head(10).to_string())

## Матрица ОКВЭД×ОКВЭД

Агрегация транзакционных потоков по парам отраслей (ОКВЭД отправителя × ОКВЭД получателя):
- **total_turnover** — суммарный оборот между отраслями
- **edge_count** — количество транзакционных рёбер
- **avg_amount** — средняя сумма транзакции

Узлы с ОКВЭД = "00" (физлица, не имеющие отраслевого кода) исключаются.

In [ ]:
okved_matrix = build_okved_matrix(G)

if not okved_matrix.empty:
    print(f'OKVED matrix: {len(okved_matrix)} pairs')
    print(f'Total cross-industry turnover: {okved_matrix["total_turnover"].sum():,.0f}')
    print(f'\nTop-15 OKVED pairs by turnover:')
    display(okved_matrix.nlargest(15, 'total_turnover'))
else:
    print('OKVED matrix is empty (no transaction edges between nodes with OKVED codes).')

## Тепловая карта межотраслевых потоков

Визуализация матрицы как heatmap. Тёмные ячейки = большой оборот между отраслями.

**Как читать**: строка = отрасль отправителя, столбец = отрасль получателя. Диагональ = внутриотраслевые платежи.

In [ ]:
if not okved_matrix.empty:
    # Pivot для heatmap
    pivot = okved_matrix.pivot_table(
        index='okved_source', columns='okved_target',
        values='total_turnover', fill_value=0,
    )

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(np.log1p(pivot.values), cmap='YlOrRd', aspect='auto')

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(pivot.index, fontsize=9)

    ax.set_xlabel('OKVED target (receiver)', fontsize=11)
    ax.set_ylabel('OKVED source (sender)', fontsize=11)
    ax.set_title('Cross-Industry Transaction Flows (log scale)', fontsize=13)

    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('log(1 + total_turnover)', fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print('No data for heatmap.')

## ОКВЭД-разнообразие узлов

Для каждого узла вычисляется:
- **okved_diversity_count** — количество уникальных отраслей контрагентов
- **okved_diversity_entropy** — энтропия Шеннона распределения по отраслям (выше = равномернее распределены контрагенты по отраслям)
- **is_cross_industry_hub** — True, если узел в топ-10% по diversity_count

**Интерпретация**: высокий diversity_count + высокая entropy = компания работает со многими отраслями примерно поровну (возможный торговый дом или посредник).

In [ ]:
G = compute_okved_diversity(G)

# Собираем метрики
diversity_data = []
for node, data in G.nodes(data=True):
    diversity_data.append({
        'client_uk': node,
        'name': data.get('name', str(node)),
        'okved_code': data.get('okved_code', '00'),
        'okved_diversity_count': data.get('okved_diversity_count', 0),
        'okved_diversity_entropy': data.get('okved_diversity_entropy', 0.0),
        'is_cross_industry_hub': data.get('is_cross_industry_hub', False),
    })

div_df = pd.DataFrame(diversity_data).set_index('client_uk')

hub_count = div_df['is_cross_industry_hub'].sum()
print(f'Cross-industry hubs: {hub_count} / {len(div_df)}')
print(f'\nDiversity distribution:')
print(div_df['okved_diversity_count'].describe())

## Кросс-отраслевые хабы

Компании, которые работают с контрагентами из наибольшего числа отраслей. Это могут быть:
- **Торговые дома / дистрибуторы** — легитимные компании с широким спектром поставщиков/покупателей
- **Финансовые посредники** — компании, оказывающие услуги разным отраслям
- **Подозрительные узлы** — если компания мелкая, но работает с 10+ отраслями, это необычно

In [ ]:
# Топ-20 по ОКВЭД-разнообразию
top_diverse = div_df.nlargest(20, 'okved_diversity_count')
print('Top-20 nodes by OKVED diversity:')
display(top_diverse[['name', 'okved_code', 'okved_diversity_count',
                      'okved_diversity_entropy', 'is_cross_industry_hub']])

In [ ]:
# Распределение diversity_count
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(div_df['okved_diversity_count'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('OKVED Diversity Count')
axes[0].set_ylabel('Number of Nodes')
axes[0].set_title('Distribution of OKVED Diversity')
axes[0].axvline(div_df['okved_diversity_count'].quantile(0.9), color='red',
                linestyle='--', label='Top 10% threshold')
axes[0].legend()

# Entropy vs Count scatter
axes[1].scatter(div_df['okved_diversity_count'], div_df['okved_diversity_entropy'],
                alpha=0.5, s=20)
hubs = div_df[div_df['is_cross_industry_hub']]
axes[1].scatter(hubs['okved_diversity_count'], hubs['okved_diversity_entropy'],
                alpha=0.8, s=40, color='red', label='Cross-industry hubs')
axes[1].set_xlabel('OKVED Diversity Count')
axes[1].set_ylabel('OKVED Diversity Entropy')
axes[1].set_title('Diversity Count vs Entropy')
axes[1].legend()

plt.tight_layout()
plt.show()

## Сохранение результатов

In [ ]:
# Сохраняем обновлённый граф с ОКВЭД-метриками
with open(config.OUTPUT_FILTERED_GRAPH_PICKLE, 'wb') as f:
    pickle.dump(G, f)
print(f'Graph with OKVED metrics saved: {config.OUTPUT_FILTERED_GRAPH_PICKLE}')

# Сохраняем ОКВЭД-матрицу как Parquet
if not okved_matrix.empty:
    okved_path = os.path.join(config.DATA_DIR, 'okved_matrix.parquet')
    okved_matrix.to_parquet(okved_path, index=False)
    print(f'OKVED matrix saved: {okved_path}')

---

## Глоссарий терминов

| Термин | Описание |
|--------|----------|
| **ОКВЭД** | Общероссийский классификатор видов экономической деятельности; двузначный код отрасли (напр. 46 = оптовая торговля, 62 = ИТ, 41 = строительство) |
| **Матрица ОКВЭД×ОКВЭД** | Таблица: строка = отрасль отправителя, столбец = отрасль получателя, значение = суммарный оборот |
| **Heatmap (тепловая карта)** | Визуализация матрицы цветом: чем темнее ячейка — тем больше оборот между отраслями |
| **okved_diversity_count** | Количество уникальных отраслей среди контрагентов узла |
| **okved_diversity_entropy** | Энтропия Шеннона: насколько равномерно распределены контрагенты по отраслям (0 = одна отрасль, высокое = много равномерно) |
| **is_cross_industry_hub** | True = компания в топ-10% по количеству отраслей контрагентов |
| **Кросс-отраслевой хаб** | Компания, работающая с контрагентами из аномально большого числа отраслей |

---

**Следующий шаг**: Откройте `06_behavioral_segmentation.ipynb` для поведенческой сегментации.